# 05 — External Dataset Bonus: Shark Attacks vs. International Tourism Arrivals

- **Owner:** Romain Thepaut (`@RomThpt`)
- **Course Rubric:** Task 6 — External Dataset `[+1 extra point]`
- **External Source:** World Bank Indicator `ST.INT.ARVL` — *International tourism, number of arrivals* (1995-2020, 266 countries/regions)

In [1]:
# Import libraries and project functions
import os
import sys
import warnings
import pandas as pd
import plotly.express as px

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_shark_data, load_tourism_data, clean_shark_data, clean_and_merge_tourism
from src.indicators import query_tourism_enrichment

warnings.filterwarnings('ignore')
clean_sharks = clean_shark_data(load_shark_data('../data/attacks.csv'))
raw_tourism = load_tourism_data('../data/tourism_arrivals.csv')
merged_tourism = clean_and_merge_tourism(clean_sharks, raw_tourism)
print(f'Merged dataset ready: {len(merged_tourism)} country-year observations ({merged_tourism["Year"].min()}-{merged_tourism["Year"].max()}).')

Merged dataset ready: 355 country-year observations (1995-2018).


## External Bonus Indicator — Tourism-Normalized Shark Attack Risk `[+1 pt]`

### Indicator Explanation
- **Calculation:** Joined annual shark attack counts with World Bank inbound tourist arrivals on `(Country_clean, Year)` across **1995-2018**, then calculated:
  $$\text{Attacks per 1 Million Tourists} = \frac{\text{Annual Attacks}}{\text{Inbound Tourist Arrivals}} \times 1{,}000{,}000$$
  Computed both **Pearson ($r$)** and **Spearman ($\rho$)** correlation coefficients across country-year observations.
- **What it represents:** Separates raw encounter volume (driven by sheer tourist traffic) from relative per-visitor encounter risk.
- **Interpretation:**
  - A strong positive correlation (**Pearson $r = 0.5967, p < 0.001$**) confirms that nations welcoming massive tourism flows naturally experience more total shark encounters over time.
  - Normalizing by tourist arrivals completely **reverses the country risk ranking**: the United States leads raw attacks (1,180) yet exhibits a very low exposure rate (**0.39 attacks / 1M tourists**), whereas **Australia (3.14)** is ~8x higher and **New Caledonia (18.31)** displays the highest tourist-adjusted risk among top affected destinations.

In [2]:
# Execute External Tourism Bonus analysis
bonus = query_tourism_enrichment(merged_tourism)

print(f"Global Pearson correlation (Arrivals vs. Attacks): r = {bonus['pearson_corr']:.4f} (p = {bonus['pearson_pval']:.4e})")
print(f"Global Spearman rank correlation: rho = {bonus['spearman_corr']:.4f} (p = {bonus['spearman_pval']:.4e})")

print('\n--- Top 8 Countries: Raw Attacks vs. Attacks per 1 Million Tourists (1995-2018) ---')
display(bonus['country_comparison'])
bonus['fig_comparison'].show()
bonus['fig_dual_axis'].show()

Global Pearson correlation (Arrivals vs. Attacks): r = 0.5967 (p = 1.3065e-35)
Global Spearman rank correlation: rho = 0.3423 (p = 3.3972e-11)

--- Top 8 Countries: Raw Attacks vs. Attacks per 1 Million Tourists (1995-2018) ---


,Country_clean,Years_Observed,Total_Attacks,Total_Tourist_Arrivals,attacks_per_1M_tourists
0,United States,24,1180,3.039728e+09,0.388
1,Australia,24,433,1.379510e+08,3.139
2,South Africa,24,173,2.277430e+08,0.760
3,Brazil,23,78,1.147470e+08,0.680
4,"Bahamas, The",22,59,1.095840e+08,0.538
5,New Zealand,19,51,4.536700e+07,1.124
6,New Caledonia,16,31,1.693000e+06,18.311
7,Mexico,12,28,1.091718e+09,0.026


In [3]:
# Additional cross-country view: annual evolution of attacks per 1M tourists for Australia, South Africa, and US
focus_countries = ['Australia', 'South Africa', 'United States', 'New Zealand']
trend_df = bonus['enriched_df'][bonus['enriched_df']['Country_clean'].isin(focus_countries)].copy()
fig_rate_trend = px.line(
    trend_df,
    x='Year',
    y='attacks_per_1M_tourists',
    color='Country_clean',
    markers=True,
    labels={'attacks_per_1M_tourists': 'Attacks per 1M Tourists', 'Country_clean': 'Country'},
    title='Annual Evolution of Tourism-Normalized Shark Risk Rate (1995-2018)',
)
fig_rate_trend.update_layout(template='plotly_white')
fig_rate_trend.show()